# Chapter 27: Well Network Optimization

This notebook demonstrates optimization of a multi-well gathering network. Four wells with
different reservoir pressures and productivity indices are connected to a common manifold.
We optimize choke settings to maximize total production while respecting manifold and
pipeline pressure constraints.

**Key Concepts:**
- Multi-well gathering network modeling
- Choke valve pressure drop and flow control
- Well inflow performance (IPR)
- Network optimization for maximum total rate

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import importlib, subprocess, sys

try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False)
    ns = neqsim_classes(ns)
    NEQSIM_MODE = "devtools"
    print("NeqSim loaded via devtools (local dev mode)")
except Exception:
    NEQSIM_MODE = "pip"

# Always ensure jneqsim is available (works in both modes)
try:
    import neqsim
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])

from neqsim import jneqsim
print(f"NeqSim ready (mode: {NEQSIM_MODE})")

# Common class shortcuts for convenience
SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
SystemPrEos = jneqsim.thermo.system.SystemPrEos
SystemSrkCPAstatoil = jneqsim.thermo.system.SystemSrkCPAstatoil
ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations

# Process equipment
Stream = jneqsim.process.equipment.stream.Stream
Separator = jneqsim.process.equipment.separator.Separator
ThreePhaseSeparator = jneqsim.process.equipment.separator.ThreePhaseSeparator
Compressor = jneqsim.process.equipment.compressor.Compressor
Cooler = jneqsim.process.equipment.heatexchanger.Cooler
Heater = jneqsim.process.equipment.heatexchanger.Heater
HeatExchanger = jneqsim.process.equipment.heatexchanger.HeatExchanger
Mixer = jneqsim.process.equipment.mixer.Mixer
Splitter = jneqsim.process.equipment.splitter.Splitter
ThrottlingValve = jneqsim.process.equipment.valve.ThrottlingValve
Pump = jneqsim.process.equipment.pump.Pump
Expander = jneqsim.process.equipment.expander.Expander
Recycle = jneqsim.process.equipment.util.Recycle
ProcessSystem = jneqsim.process.processmodel.ProcessSystem

NeqSim project root: C:\Users\ESOL\Documents\GitHub\neqsim2
Classpath:
  1. C:\Users\ESOL\Documents\GitHub\neqsim2\target\classes
  2. C:\Users\ESOL\Documents\GitHub\neqsim2\src\main\resources
  3. C:\Users\ESOL\Documents\GitHub\neqsim2\target\neqsim-3.7.0.jar



JVM started: C:\Users\ESOL\graalvm\graalvm-jdk-25.0.1+8.1\bin\server\jvm.dll
Ready — call neqsim_classes(ns) to import classes


All NeqSim classes imported OK
NeqSim loaded via devtools (local dev mode)
NeqSim ready (mode: devtools)


## 27.1 Define Well Fluids and Network

Each well produces gas at different reservoir conditions. We model four wells with
choke valves flowing into a common mixer (manifold), followed by a pipeline.

In [2]:
from neqsim import jneqsim

# --- Well parameters ---
well_data = [
    {"name": "Well A", "pres": 250.0, "temp": 90.0, "rate": 30000.0},
    {"name": "Well B", "pres": 220.0, "temp": 85.0, "rate": 25000.0},
    {"name": "Well C", "pres": 200.0, "temp": 80.0, "rate": 20000.0},
    {"name": "Well D", "pres": 180.0, "temp": 75.0, "rate": 15000.0},
]

# --- Common fluid composition ---
def create_well_fluid(temp_c, pres_bara):
    f = jneqsim.thermo.system.SystemSrkEos(273.15 + temp_c, pres_bara)
    f.addComponent("nitrogen", 0.01)
    f.addComponent("CO2", 0.015)
    f.addComponent("methane", 0.85)
    f.addComponent("ethane", 0.06)
    f.addComponent("propane", 0.03)
    f.addComponent("n-butane", 0.02)
    f.addComponent("n-pentane", 0.01)
    f.addComponent("water", 0.005)
    f.setMixingRule("classic")
    return f

# --- Build network ---
manifold_pressure = 80.0  # bara

streams = []
chokes = []
process = jneqsim.process.processmodel.ProcessSystem()

for wd in well_data:
    fluid = create_well_fluid(wd["temp"], wd["pres"])
    stream = jneqsim.process.equipment.stream.Stream(wd["name"], fluid)
    stream.setFlowRate(wd["rate"], "kg/hr")
    stream.setTemperature(wd["temp"], "C")
    stream.setPressure(wd["pres"], "bara")

    choke = jneqsim.process.equipment.valve.ThrottlingValve(wd["name"] + " Choke", stream)
    choke.setOutletPressure(manifold_pressure)

    streams.append(stream)
    chokes.append(choke)
    process.add(stream)
    process.add(choke)

# --- Mixer (manifold) ---
mixer = jneqsim.process.equipment.mixer.Mixer("Production Manifold")
for choke in chokes:
    mixer.addStream(choke.getOutletStream())

process.add(mixer)
process.run()

total_rate = mixer.getOutletStream().getFlowRate("kg/hr")
print(f"Total production at manifold: {total_rate:.0f} kg/hr")
print(f"Manifold temperature: {mixer.getOutletStream().getTemperature('C'):.1f} °C")
print(f"Manifold pressure: {mixer.getOutletStream().getPressure('bara'):.1f} bara")

for i, wd in enumerate(well_data):
    print(f"  {wd['name']}: {streams[i].getFlowRate('kg/hr'):.0f} kg/hr, "
          f"dP choke = {wd['pres'] - manifold_pressure:.0f} bar")

Total production at manifold: 90000 kg/hr
Manifold temperature: 54.2 °C
Manifold pressure: 80.0 bara
  Well A: 30000 kg/hr, dP choke = 170 bar
  Well B: 25000 kg/hr, dP choke = 140 bar
  Well C: 20000 kg/hr, dP choke = 120 bar
  Well D: 15000 kg/hr, dP choke = 100 bar


## 27.2 Manifold Pressure Sensitivity

We sweep the manifold pressure and observe how individual well rates and total production change.
Lower manifold pressure allows more flow but increases choke erosion risk.

In [3]:
# --- Sweep manifold pressure ---
manifold_pressures = np.linspace(50, 150, 20)
well_rates = {wd["name"]: [] for wd in well_data}
total_rates = []

for p_man in manifold_pressures:
    for i, choke in enumerate(chokes):
        # Only set outlet pressure if below well reservoir pressure
        if p_man < well_data[i]["pres"]:
            choke.setOutletPressure(float(p_man))
    process.run()

    total = 0
    for i, wd in enumerate(well_data):
        rate = streams[i].getFlowRate("kg/hr")
        well_rates[wd["name"]].append(rate)
        total += rate
    total_rates.append(total)

# Reset
for choke in chokes:
    choke.setOutletPressure(manifold_pressure)
process.run()

# --- Plot ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Individual well rates
colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']
for i, wd in enumerate(well_data):
    ax1.plot(manifold_pressures, np.array(well_rates[wd["name"]]) / 1000,
             '-o', markersize=3, color=colors[i], label=wd["name"])

ax1.set_xlabel('Manifold Pressure (bara)', fontsize=12)
ax1.set_ylabel('Well Rate (1000 kg/hr)', fontsize=12)
ax1.set_title('Individual Well Rates vs Manifold Pressure', fontsize=13)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Total production
ax2.plot(manifold_pressures, np.array(total_rates) / 1000, 'k-o', markersize=4, linewidth=2)
ax2.axvline(x=manifold_pressure, color='red', linestyle='--', label=f'Design ({manifold_pressure} bara)')
ax2.set_xlabel('Manifold Pressure (bara)', fontsize=12)
ax2.set_ylabel('Total Production (1000 kg/hr)', fontsize=12)
ax2.set_title('Total Production vs Manifold Pressure', fontsize=13)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.suptitle('Chapter 27: Well Network — Manifold Pressure Sensitivity', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("../figures/ch27_manifold_pressure_sweep.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved: ../figures/ch27_manifold_pressure_sweep.png")

Figure saved: ../figures/ch27_manifold_pressure_sweep.png


C:\Users\ESOL\AppData\Local\Temp\ipykernel_42140\1863620215.py:52: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 27.3 Choke Opening Sensitivity

We simulate the effect of partially closing each well's choke (reducing its outlet pressure
further) to understand how individual well choke adjustments affect the total network rate.

In [4]:
# --- Choke sensitivity: vary one well's choke while others remain at design ---
choke_dp_range = np.linspace(0, 100, 20)  # additional dP beyond base

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for idx, (wd, choke, stream, ax) in enumerate(zip(well_data, chokes, streams, axes.flat)):
    rates_this_well = []
    rates_total = []

    for dp_extra in choke_dp_range:
        p_out = manifold_pressure - dp_extra
        if p_out < 10.0:  # minimum pressure
            p_out = 10.0
        choke.setOutletPressure(float(p_out))
        process.run()
        rates_this_well.append(stream.getFlowRate("kg/hr"))
        rates_total.append(mixer.getOutletStream().getFlowRate("kg/hr"))

    # Reset choke
    choke.setOutletPressure(manifold_pressure)

    ax.plot(choke_dp_range, np.array(rates_this_well) / 1000, 'b-', linewidth=2, label=f'{wd["name"]} rate')
    ax.set_xlabel('Additional Choke dP (bar)', fontsize=10)
    ax.set_ylabel('Rate (1000 kg/hr)', fontsize=10)
    ax.set_title(f'{wd["name"]} — Choke Sensitivity', fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)

process.run()  # final reset

plt.suptitle('Chapter 27: Choke Opening Sensitivity per Well', fontsize=14)
plt.tight_layout()
plt.savefig("../figures/ch27_choke_sensitivity.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved: ../figures/ch27_choke_sensitivity.png")

Figure saved: ../figures/ch27_choke_sensitivity.png


C:\Users\ESOL\AppData\Local\Temp\ipykernel_42140\731223873.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 27.4 Summary

Well network optimization in NeqSim demonstrates:

1. **Individual well rates** are controlled by the pressure drop across the choke valve
2. **Manifold pressure** directly affects total production — lower pressure increases flow but may cause operational issues
3. **Wells with higher reservoir pressure** contribute more flow and have greater choke flexibility
4. **Network optimization** balances individual well constraints against total production targets

In practice, real-time optimization of choke settings is a key activity for production engineers,
especially when coupled with well test data and decline curve analysis.